In [ ]:
import mlflow
import numpy as np
import pandas as pd

import dataprocessing as dc


In [ ]:
#Uncomment to create the csv

# df = pd.read_table('data/SMSSpamCollection', header=None, names=['Category', 'Message'])

# # Save to CSV format
# df.to_csv('data/SMSSpamCollection.csv', index=False, encoding='utf-8')

In [ ]:
datafile = "SMSSpamCollection.csv"

df = pd.read_csv(f"data/{datafile}")
newdata = dc.Data_class(data = df, )
df = newdata.data
df["Category"] = np.where(df['Category'] == 'spam', -1, 1)
df.head(2)

In [ ]:
experiment_name = "Default"
"""
metrics

cv_best_spam_f1
cv_mean_spam_f1
cv_std_spam_f1
test_roc_auc
test_accuracy
precision
recall
f1 score
weighted f1 score
"""
metric_to_sort_by = "cv_best_spam_f1"
ascending = False

experiment = mlflow.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id

runs_df = mlflow.search_runs(experiment_ids=[experiment_id])
runs_df = runs_df[runs_df["status"] == "FINISHED"]

best_run = (
    runs_df
    .sort_values(by=f"metrics.{metric_to_sort_by}", ascending=ascending)
    .iloc[0]
)

best_run_id = best_run["run_id"]
print("Best run ID:", best_run_id)
print("Run name:", best_run["tags.mlflow.runName"])
print(f"Best {metric_to_sort_by}:", best_run[f"metrics.{metric_to_sort_by}"])

In [ ]:
model_uri = f"runs:/{best_run_id}/best_model"
y = df["Category"]
X_new = df.drop(
    ["Category", "Message", "word_count"],
    axis = 1
)
loaded_model = mlflow.sklearn.load_model(model_uri)

predictions = loaded_model.predict(X_new)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y, predictions)
print(f"Accuracy: {accuracy:.2%}")